# Xori Hori — Llama 3.2 3B LoRA

Этот notebook переобучает Xori LoRA на `meta-llama/Llama-3.2-3B-Instruct` вместо старого Qwen 0.5B. Цель — получить Cloudflare-compatible adapter: `adapter_config.json` + `adapter_model.safetensors`.

Для Cloudflare Workers AI адаптер должен быть совместим с базовой моделью, иметь небольшой размер и подходящий rank; здесь ставим `r=8` и не используем quantization при обучении. Cloudflare сейчас указывает Llama 3.2 3B как модель с поддержкой LoRA. 

In [ ]:
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.16' 'accelerate>=1.2' datasets huggingface_hub safetensors

import os, json, shutil, subprocess
from pathlib import Path
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Нужен GPU runtime. В Colab: Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from huggingface_hub import login, whoami

# Не вставляй токен в чат. Введи его только в это поле Colab.
HF_TOKEN = input('Hugging Face token: ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN пустой.')
login(token=HF_TOKEN, add_to_git_credential=False)
me = whoami(token=HF_TOKEN)
print('HF:', me.get('name') or me.get('username'))


In [ ]:
SAFE_ROOT = '/tmp'
os.chdir(SAFE_ROOT)
REPO_URL = 'https://github.com/xoristalin-dotcom/Xori-.git'
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working')
WORK = WORK_ROOT / 'Xori-'
if WORK.exists():
    shutil.rmtree(WORK, ignore_errors=True)
p = subprocess.run(['git','clone','--depth','1',REPO_URL,str(WORK)],cwd=SAFE_ROOT,capture_output=True,text=True)
print(p.stdout); print(p.stderr)
if p.returncode != 0:
    raise RuntimeError('Git clone failed')
os.chdir(WORK)
print('Repo:', WORK)


In [ ]:
pairs = []
seed = Path('hori_sft_seed.jsonl')
if seed.exists():
    for line in seed.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line)
            if row.get('user') and row.get('assistant'):
                pairs.append((row['user'].strip(), row['assistant'].strip()))
training = Path('hori_training.json')
if training.exists():
    data = json.loads(training.read_text(encoding='utf-8'))
    for row in data.get('examples', []):
        if row.get('approved') and row.get('user') and row.get('assistant'):
            pairs.append((row['user'].strip(), row['assistant'].strip()))
seen=set(); clean=[]
for pair in pairs:
    if pair not in seen:
        seen.add(pair); clean.append(pair)
pairs=clean
print('Training pairs:', len(pairs))
if len(pairs) < 30:
    raise RuntimeError('Слишком мало примеров. Проверь hori_sft_seed.jsonl и hori_training.json.')


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE = 'meta-llama/Llama-3.2-3B-Instruct'
MAX_LEN = 1024
SYSTEM = (
    'Ты — Хори Кёко из Horimiya. '
    'Отвечай по-русски естественно, прямо и по-человечески. '
    'Не копируй реплики из произведения. Не выдумывай мысли, действия или факты о собеседнике. '
    'Не форсируй дружбу или романтику. Собеседник сам управляет своими действиями и словами. '
    'Сохраняй характер Хори: ответственная, заботливая, прямая, иногда вспыльчивая, упрямая и с чувством юмора. '
    'Обычно отвечай 2–5 предложениями и не задавай больше одного вопроса.'
)
tokenizer = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer OK:', tokenizer.__class__.__name__)


In [ ]:
def encode_pair(user_text, assistant_text):
    messages = [{'role':'system','content':SYSTEM},{'role':'user','content':user_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(assistant_text, add_special_tokens=False)['input_ids']
    eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    ids = prompt_ids + answer_ids + eos
    labels = [-100] * len(prompt_ids) + answer_ids + eos
    if len(ids) > MAX_LEN:
        # Сохраняем конец ответа, но не режем его случайно посередине без необходимости.
        ids = ids[-MAX_LEN:]
        labels = labels[-MAX_LEN:]
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

rows=[encode_pair(u,a) for u,a in pairs]
dataset=Dataset.from_list(rows)
print(dataset)
print('Example length:', len(dataset[0]['input_ids']))


In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16
print('dtype:', dtype)

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    token=HF_TOKEN,
    torch_dtype=dtype,
)
model.config.use_cache = False

# r=8 — консервативный rank, подходящий для Cloudflare LoRA.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

OUTPUT_DIR = WORK_ROOT / 'xori_llama32_3b_lora'
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    optim='adamw_torch',
    remove_unused_columns=False,
)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,padding=True,label_pad_token_id=-100,return_tensors='pt')
trainer = Trainer(model=model,args=args,train_dataset=dataset,data_collator=collator)
print('Starting training...')
trainer.train()


In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# Cloudflare требует точные имена и model_type.
cfg_path = OUTPUT_DIR / 'adapter_config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
cfg['model_type'] = 'llama'
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')

adapter = OUTPUT_DIR / 'adapter_model.safetensors'
print('Output:', OUTPUT_DIR)
print('adapter_model.safetensors MB:', round(adapter.stat().st_size/1024/1024,2))
print('rank:', cfg.get('r'))
print('model_type:', cfg.get('model_type'))
assert adapter.exists()
assert cfg.get('model_type') == 'llama'
assert int(cfg.get('r',99)) <= 32
print('Cloudflare adapter structure check: OK')


In [ ]:
# Скачаем готовый adapter как ZIP из Colab.
import shutil
zip_path = shutil.make_archive(str(WORK_ROOT / 'xori_llama32_3b_lora'), 'zip', root_dir=str(OUTPUT_DIR))
print(zip_path)
print('В ZIP должны быть минимум: adapter_config.json и adapter_model.safetensors')


## Что делать после обучения

Не загружай токен в GitHub и не отправляй его в чат. Сохрани локально только `adapter_config.json` и `adapter_model.safetensors`. Cloudflare сейчас позволяет создать fine-tune и загрузить эти два файла, после чего вызывать LoRA через параметр `lora`. Для Llama 3.2 3B Workers AI поддерживает LoRA. 